## Shallow and Deep Autoencoders with MNIST

In the example a shallow and a deep autoencoder network is trained with the MNIST dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm.notebook import trange

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

### Prepare the Data

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

In [ ]:
train_x = train_dataset.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_dataset.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255

In [ ]:
train_x = train_x.to(device)
train_dataset = TensorDataset(train_x)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_dataset = TensorDataset(test_x)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, 
                                          shuffle=False, drop_last=True)

### Build the Autoencoders

In [ ]:
class ShallowAutoencoder(nn.Module):
    def __init__(self, hidden_width=32):
        super(ShallowAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28*28, hidden_width),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_width, 28*28),
            nn.Linear(28*28, 28*28)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
class DeepAutoencoder(nn.Module):
    def __init__(self):
        super(DeepAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Linear(28*28, 28*28)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

### Train the models

In [ ]:
no_epochs = 50
batch_size = 256
criterion = nn.MSELoss()

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'val_loss': []}

    for epoch in trange(num_epochs):
        model.train()
        train_loss = 0
        for [images] in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, images)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss = train_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for [images] in test_loader:
                outputs = model(images)
                loss = criterion(outputs, images)
                val_loss += loss.item() * images.size(0)

        val_loss = val_loss / len(test_loader.dataset)
        history['val_loss'].append(val_loss)

        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

    return history

In [ ]:
shallow_auto = ShallowAutoencoder().to(device)
shallow_history = train_model(shallow_auto, train_loader, test_loader, no_epochs)

In [ ]:
deep_auto = DeepAutoencoder().to(device)
deep_history = train_model(deep_auto, train_loader, test_loader, no_epochs)

### Plot Validation Loss

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, shallow_history['val_loss'], label='shallow', markevery=10)
ax.plot(epochs, deep_history['val_loss'], label='deep', markevery=10)

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Loss')
ax.legend()
fig.savefig('TestShallowDeepAuto.png', dpi=300, bbox_inches='tight')

### Display Images and the 2 Reconstructions

In [ ]:
def reconstruct(model, test_loader):
    model.eval()
    reconstructions = []
    with torch.no_grad():
        for [images] in test_loader:
            outputs = model(images)
            reconstructions.append(outputs.cpu().numpy())

    return np.concatenate(reconstructions, axis=0)

In [ ]:
shallow_reconstruction = reconstruct(shallow_auto, test_loader)
deep_reconstruction = reconstruct(deep_auto, test_loader)

In [ ]:
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Display original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(test_loader.dataset[i][0].view(28, 28).cpu().detach().numpy(), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display shallow
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(shallow_reconstruction[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display deep
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(deep_reconstruction[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.savefig('autoreconstruct.png', dpi=300, bbox_inches='tight')